# Train Model
Train any sklearn/xgboost model

In [1]:
import os
import json
import joblib
import pathlib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [2]:
MODEL_NAME         = "churn_random_forest_v1"
MODEL_VERSION      = "v1"
TASK_TYPE          = "classification"
TEST_SIZE          = 0.2
RANDOM_STATE       = 42

REPO_ROOT          = r"D:\ModelGuard AI"

MODELS_SAVED_DIR   = os.path.join(REPO_ROOT, "models", "saved")
MODELS_VERSION_DIR = os.path.join(REPO_ROOT, "models", "versions")
DATA_RAW_DIR       = os.path.join(REPO_ROOT, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(REPO_ROOT, "data", "processed")
DATA_REFERENCE_DIR = os.path.join(REPO_ROOT, "data", "reference")

for path in [MODELS_SAVED_DIR, MODELS_VERSION_DIR,
             DATA_RAW_DIR, DATA_PROCESSED_DIR, DATA_REFERENCE_DIR]:
    os.makedirs(path, exist_ok=True)

print(f"REPO_ROOT       → {REPO_ROOT}")
print(f"DATA_RAW_DIR    → {DATA_RAW_DIR}")
print(f"churn.csv found → {os.path.exists(os.path.join(DATA_RAW_DIR, 'churn.csv'))}")

REPO_ROOT       → D:\ModelGuard AI
DATA_RAW_DIR    → D:\ModelGuard AI\data\raw
churn.csv found → True


In [3]:
df = pd.read_csv(os.path.join(DATA_RAW_DIR, "churn.csv"))
print(f"Raw shape: {df.shape}")

df.drop(columns=["customerID"], inplace=True)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df.drop(columns=["Churn"])
y = df["Churn"]

print(f"Features  : {X.shape[1]}")
print(f"Churn rate: {round(y.mean() * 100, 1)}%")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

preprocessor = StandardScaler()
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

pd.DataFrame(X_train_scaled, columns=X.columns).to_csv(
    os.path.join(DATA_PROCESSED_DIR, "X_train.csv"), index=False
)
pd.DataFrame(X_test_scaled, columns=X.columns).to_csv(
    os.path.join(DATA_PROCESSED_DIR, "X_test.csv"), index=False
)
print("✓ Processed data saved")

Raw shape: (7043, 21)
Features  : 30
Churn rate: 26.5%
Train: 5634  |  Test: 1409


C:\Users\Soul\AppData\Local\Temp\ipykernel_7336\3052030227.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


✓ Processed data saved


In [4]:
model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)

y_pred   = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred, target_names=["Stays", "Churns"]))

Accuracy: 0.7857
              precision    recall  f1-score   support

       Stays       0.83      0.89      0.86      1035
      Churns       0.62      0.49      0.55       374

    accuracy                           0.79      1409
   macro avg       0.73      0.69      0.70      1409
weighted avg       0.77      0.79      0.78      1409



In [6]:
joblib.dump(model,        os.path.join(MODELS_SAVED_DIR, "model.pkl"))
joblib.dump(preprocessor, os.path.join(MODELS_SAVED_DIR, "preprocessor.pkl"))
joblib.dump(model,        os.path.join(MODELS_VERSION_DIR, f"model_{MODEL_VERSION}.pkl"))

print("✓ model.pkl saved")
print("✓ preprocessor.pkl saved")

# Fix: convert all boolean columns to float before stats
X_train_stats = X_train.astype(float)

reference_stats = {}
for col in X_train_stats.columns:
    col_data = X_train_stats[col].values
    reference_stats[col] = {
        "mean"  : float(np.mean(col_data)),
        "std"   : float(np.std(col_data)),
        "min"   : float(np.min(col_data)),
        "max"   : float(np.max(col_data)),
        "median": float(np.median(col_data)),
        "p25"   : float(np.percentile(col_data, 25)),
        "p75"   : float(np.percentile(col_data, 75)),
    }

with open(os.path.join(DATA_REFERENCE_DIR, "reference_stats.json"), "w") as f:
    json.dump(reference_stats, f, indent=2)

X_train.to_csv(
    os.path.join(DATA_REFERENCE_DIR, "X_train_reference.csv"), index=False
)

with open(os.path.join(DATA_REFERENCE_DIR, "feature_names.json"), "w") as f:
    json.dump(list(X.columns), f, indent=2)

metadata = {
    "model_name"    : MODEL_NAME,
    "model_version" : MODEL_VERSION,
    "task_type"     : TASK_TYPE,
    "feature_names" : list(X.columns),
    "n_features"    : len(X.columns),
    "train_samples" : len(X_train),
    "test_accuracy" : round(accuracy, 4),
    "trained_at"    : pd.Timestamp.now().isoformat(),
}

with open(os.path.join(DATA_REFERENCE_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("✓ reference_stats.json saved")
print("✓ X_train_reference.csv saved")
print("✓ feature_names.json saved")
print("✓ model_metadata.json saved")
print(f"\n{'─'*40}")
print(f"  Phase 0 complete.")
print(f"  Accuracy : {accuracy:.4f}")
print(f"  Ready for Phase 1.")
print(f"{'─'*40}")

✓ model.pkl saved
✓ preprocessor.pkl saved
✓ reference_stats.json saved
✓ X_train_reference.csv saved
✓ feature_names.json saved
✓ model_metadata.json saved

────────────────────────────────────────
  Phase 0 complete.
  Accuracy : 0.7857
  Ready for Phase 1.
────────────────────────────────────────
